# Implement the Gymnasium Environment

In this notebook, you will implement the core methods of a [Gymnasium](https://gymnasium.farama.org/) environment for the battery scheduling problem.

## The Gymnasium Interface

Gymnasium provides a standard interface for RL environments. Every environment has two key methods:

- **`reset()`** — Start a new episode. Returns `(observation, info)`.
- **`step(action)`** — Take one action. Returns `(observation, reward, terminated, truncated, info)`.

The agent-environment loop looks like this:

```python
obs, info = env.reset()

while not terminated:
    action = agent.choose_action(obs)
    obs, reward, terminated, truncated, info = env.step(action)
```

| Return value | Description |
|-------------|-------------|
| `observation` | What the agent sees (numpy array, normalized to [0, 1]) |
| `reward` | Scalar feedback signal (higher = better) |
| `terminated` | `True` when the episode ends naturally |
| `truncated` | `True` if cut short externally (we don't use this) |
| `info` | Debug dictionary with human-readable state |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from envs.battery_env import BatteryStorageEnv

## Your Task

Open the file **`01_workshop/envs/battery_env.py`** and implement the 4 methods below the line:

```python
# =========================================================================
# METHODS FOR PARTICIPANTS TO IMPLEMENT
# =========================================================================
```

Everything above that line is already implemented for you (data loading, action/observation spaces, forecasting, degradation model).

### Implementation Order

| # | Method | What it does | Difficulty |
|---|--------|-------------|------------|
| 1 | `_calculate_reward()` | Compute the cost of grid electricity | Easy (3 lines) |
| 2 | `_get_obs()` | Build the observation array | Medium |
| 3 | `reset()` | Initialize a new episode | Medium |
| 4 | `step()` | Process one action and update state | Harder |

After implementing each method, come back to this notebook and run the corresponding test cell.

## Method 1: `_calculate_reward()`

The reward tells the RL agent how good its action was. In our case, the reward is the **negative cost** of electricity purchased from the grid:

```
grid_energy = load + charge_power
```

- **Charging** (`charge_power > 0`): We buy extra electricity → higher grid cost
- **Discharging** (`charge_power < 0`): Battery offsets load → lower grid cost
- **Can't sell to grid**: If discharge exceeds load, the excess is wasted → clamp `grid_energy` to 0

The reward is negative because RL **maximizes** reward, but we want to **minimize** cost:

```
reward = -(grid_energy * price)
```

**Go implement `_calculate_reward()` now, then run the cell below.**

In [2]:
# Reload to pick up your changes
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)
from envs.battery_env import BatteryStorageEnv

env = BatteryStorageEnv()

# Test 1: Charging — grid supplies load (1.0) + charge (1.5) = 2.5 kWh
reward = env._calculate_reward(load=1.0, charge_power=1.5, price=0.20)
print(f"Charging:       reward = {reward:.4f}  (expected: -0.5000)")

# Test 2: Discharging — grid supplies load (1.0) - discharge (0.5) = 0.5 kWh
reward = env._calculate_reward(load=1.0, charge_power=-0.5, price=0.20)
print(f"Discharging:    reward = {reward:.4f}  (expected: -0.1000)")

# Test 3: Over-discharge — grid_energy would be negative, but clamped to 0
reward = env._calculate_reward(load=0.5, charge_power=-2.0, price=0.20)
print(f"Over-discharge: reward = {reward:.4f}  (expected:  0.0000)")

NotImplementedError: Implement this method

## Method 2: `_get_obs()`

The observation is a **normalized array** (all values in [0, 1]) that tells the RL agent about the current state. It should contain:

| Index | Value | How to compute |
|-------|-------|---------------|
| 0 | State of charge | `soc / max_capacity` |
| 1 | Battery health | `self.health` (already in [0, 1]) |
| 2 | Hour of day | `hours_of_day[current_step] / 24.0` |
| 3, 4 | Current price, load | `self._get_forecast(0)` |
| 5, 6 | Next hour price, load | `self._get_forecast(1)` |
| ... | ... | ... |

Use `self._get_forecast(h)` which returns a `(price, load)` tuple, already normalized.

**Important:** Normalize SoC with `self.max_capacity` (not `self.capacity`), so the value stays in [0, 1] even when capacity degrades.

**Go implement `_get_obs()` now, then move on to `reset()`.**

## Method 3: `reset()`

This method initializes a new episode. The steps are:

1. **`super().reset(seed=seed)`** — Must be called first! Sets up `self.np_random` for reproducible randomness.
2. **Select a random episode** from `self._available_episodes` (a `(start, end)` tuple of indices).
3. **Load episode data** into `self._current_prices`, `self._current_loads`, `self._current_hours_of_day`, `self._current_days_of_week` from the chunked arrays (e.g. `self.prices[episode_idx]`).
4. **Reset capacity** to `self.max_capacity` (in case it degraded during the previous episode).
5. **Random initial SoC** in `[0, capacity]`.
6. **Reset** `self.current_step = 0` and `self.health = 1.0`.
7. **Return** `(self._get_obs(), self._get_info())`.

Use `self.np_random` for all randomness:
- `self.np_random.integers(low, high)` — random int in [low, high)
- `self.np_random.uniform(low, high)` — random float in [low, high)

**Go implement `reset()` now, then run the test cell below.**

In [ ]:
# Reload and run tests for reset + _get_obs
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)

!uv run pytest ../../tests/test_battery_env.py -v -k "reset or info_dict or different_episodes or observation_space_shape or action_space_shape or forecast_horizon or split_no_overlap or data_chunked"

You should see **10 passed**. These tests verify that:
- Observation and action spaces have the correct shape (from `__init__`)
- `reset()` returns a valid observation with correct shape and bounds
- `reset()` initializes state correctly (SoC, step counter, health)
- Seeding produces reproducible results
- Different seeds select different episodes
- The info dict contains all expected keys

## Method 4: `step()`

This is the main simulation method. Each call processes one hour:

1. **Extract action** from the action array: `action = action[0]`
2. **Compute charge power**: `charge_power = action * self.max_charge_rate`
3. **Clip new SoC** to `[0, capacity]`: `new_soc = np.clip(self.soc + charge_power, 0, self.capacity)`
4. **Effective power** (accounts for battery limits): `charge_power_effective = new_soc - self.soc`
5. **Get current price and load** from `self._current_prices[self.current_step]`
6. **Calculate reward** using `self._calculate_reward(load, charge_power_effective, price)`
7. **Update state**: `self.soc = new_soc`, increment `self.current_step`
8. **Check termination**: `terminated = (self.current_step >= self.episode_length)`
9. **Degradation** (Level 2): If `self.enable_degradation`, call `self._apply_degradation(charge_power_effective)` and subtract `self.health_weight * health_damage` from the reward.
10. **Return** `(self._get_obs(), reward, terminated, False, self._get_info())`

**Go implement `step()` now, then run the test cell below.**

In [5]:
# Reload and run ALL tests
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)

!uv run pytest ../../tests/test_battery_env.py -v 

============================= test session starts ==============================
platform darwin -- Python 3.13.7, pytest-9.0.2, pluggy-1.6.0 -- /Users/david.goll/Documents/projects/workshop-rl2-implementation/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/david.goll/Documents/projects/workshop-rl2-implementation
configfile: pyproject.toml
plugins: anyio-4.12.1
collected 21 items                                                             

../../tests/test_battery_env.py::TestBatteryEnv::test_gymnasium_api_compliance FAILED [  4%]
../../tests/test_battery_env.py::TestBatteryEnv::test_observation_space_shape PASSED [  9%]
../../tests/test_battery_env.py::TestBatteryEnv::test_action_space_shape PASSED [ 14%]
../../tests/test_battery_env.py::TestBatteryEnv::test_reset_returns_valid_observation FAILED [ 19%]
../../tests/test_battery_env.py::TestBatteryEnv::test_reset_initializes_state FAILED [ 23%]
../../tests/test_battery_env.py::TestBatteryEnv::test_reset_seeding_reproducibili

**All 21 tests should pass.** If some tests fail, check the error messages — they usually point to exactly what's wrong.

## Sanity Check

Let's run your environment for a few steps to see it in action.

In [ ]:
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)
from envs.battery_env import BatteryStorageEnv

env = BatteryStorageEnv()
obs, info = env.reset(seed=42)

print(f"Observation shape: {obs.shape}")
print(f"Initial SoC:    {info['soc']:.2f} kWh")
print(f"Initial health: {info['health']:.2f}")
print()

for i in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Step {i+1}: action={action[0]:+.2f}  reward={reward:.4f}  soc={info['soc']:.2f}  price={info['price']:.3f}")

env.close()

## Next Steps

Your environment is ready! In the next notebook, we will use it to:
- Compare baseline policies (heuristic, MPC, LP)
- Train a PPO agent using Stable-Baselines3
- Evaluate how RL compares to classical approaches